In [1]:
import numpy as np
import pandas as pd
import os
import cv2
from PIL import Image
from torchvision import transforms
from torchvision import datasets
from torch.utils.data import DataLoader, Subset, Dataset
from sklearn.model_selection import train_test_split
import torch
import torch.nn as nn
import torch.optim as optim
from torchvision import models
from tqdm import tqdm
import random
from collections import Counter

In [2]:
#路徑處理

train_dir = "xray_data/train"
test_dir  = "xray_data/test"

print("Train exists:", os.path.exists(train_dir))
print("Test exists:", os.path.exists(test_dir))

print("Train classes:", os.listdir(train_dir)[:15])
print("Test samples:", os.listdir(test_dir)[:10])

Train exists: True
Test exists: True
Train classes: ['Atelectasis', 'Cardiomegaly', 'Consolidation', 'Edema', 'Effusion', 'Emphysema', 'Fibrosis', 'Hernia', 'Infiltration', 'Mass', 'No Finding', 'Nodule', 'Pleural_Thickening', 'Pneumonia', 'Pneumothorax']
Test samples: ['0.jpg', '1.jpg', '10.jpg', '100.jpg', '1000.jpg', '1001.jpg', '1002.jpg', '1003.jpg', '1004.jpg', '1005.jpg']


In [3]:
# 各種影像方面的處理可以選擇的

class CLAHETransform:
    def __init__(self, clip_limit=2.0, tile_grid_size=(8, 8), p=0.4):
        self.clip_limit = clip_limit
        self.tile_grid_size = tile_grid_size
        self.p = p

    def __call__(self, img):
        if random.random() > self.p:
            return img
        
        # Convert to grayscale
        img_np = np.array(img.convert("L"))

        # CLAHE local contrast enhancement
        clahe = cv2.createCLAHE(
            clipLimit=self.clip_limit,
            tileGridSize=self.tile_grid_size
        )
        img_np = clahe.apply(img_np)

        # Convert back to RGB for pretrained CNN models
        return Image.fromarray(img_np).convert("RGB")


class DenoiseTransform:
    def __init__(self, kernel_size=(3, 3), p=0.3):
        self.kernel_size = kernel_size
        self.p = p

    def __call__(self, img):
        if random.random() > self.p:
            return img
            
        # Convert to grayscale
        img_np = np.array(img.convert("L"))

        # Light Gaussian denoising
        img_np = cv2.GaussianBlur(img_np, self.kernel_size, 0)

        # Convert back to RGB
        return Image.fromarray(img_np).convert("RGB")


class GammaCorrection:
    def __init__(self, gamma=0.9, p=0.3):
        self.gamma = gamma
        self.p = p

    def __call__(self, img):
        if random.random() > self.p:
            return img
            
        # Convert image to numpy array
        img_np = np.array(img).astype(np.float32) / 255.0

        # Gamma correction
        img_np = np.power(img_np, self.gamma)
        img_np = np.clip(img_np * 255, 0, 255).astype(np.uint8)

        # Convert back to RGB
        return Image.fromarray(img_np).convert("RGB")


class SharpenTransform:
    def __init__(self, strength="light", p=0.2):
        self.strength = strength
        self.p = p

    def __call__(self, img):
        if random.random() > self.p:
            return img
            
        img_np = np.array(img)

        # Light sharpening is safer for medical images
        if self.strength == "light":
            kernel = np.array([
                [0, -0.5, 0],
                [-0.5, 3.0, -0.5],
                [0, -0.5, 0]
            ])
        else:
            kernel = np.array([
                [0, -1, 0],
                [-1, 5, -1],
                [0, -1, 0]
            ])

        img_np = cv2.filter2D(img_np, -1, kernel)
        img_np = np.clip(img_np, 0, 255).astype(np.uint8)

        return Image.fromarray(img_np).convert("RGB")


# Set probabilities for each transform
P_CLAHE = 0.4
P_DENOISE = 0.2
P_GAMMA = 0.3
P_SHARPEN = 0.1

# Parameters 可以改各種數值
CLAHE_CLIP_LIMIT = 2.0
CLAHE_TILE_GRID_SIZE = (8, 8)
GAMMA_VALUE = 0.9  
SHARPEN_STRENGTH = "light"
IMG_SIZE = 224

normalize = transforms.Normalize(
    mean=[0.485, 0.456, 0.406],
    std=[0.229, 0.224, 0.225]
)

# --- Train Transform (Stochastic Processing + Random Geometrics) ---
train_transform = transforms.Compose([
    transforms.Resize((256, 256)),
    transforms.RandomResizedCrop(IMG_SIZE, scale=(0.85, 1.0)),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomRotation(degrees=7),
    transforms.ColorJitter(brightness=0.1, contrast=0.1),
    
    # Probabilistic Medical Processing
    DenoiseTransform(kernel_size=(3, 3), p=P_DENOISE),
    CLAHETransform(clip_limit=CLAHE_CLIP_LIMIT, tile_grid_size=CLAHE_TILE_GRID_SIZE, p=P_CLAHE),
    GammaCorrection(gamma=GAMMA_VALUE, p=P_GAMMA),
    SharpenTransform(strength=SHARPEN_STRENGTH, p=P_SHARPEN),
    
    transforms.ToTensor(),
    normalize
])

# --- Validation / Test Transform (Clean baseline - No probabilistic transforms) ---
val_test_transform = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.Grayscale(num_output_channels=3), # Essential conversion back to 3-channel
    transforms.ToTensor(),
    normalize
])

In [4]:
class DatasetMultiplier(Dataset):
    """
    Wraps a PyTorch Subset or Dataset to multiply its length by a factor of (k + 1).
    """
    def __init__(self, base_dataset, k=1):
        self.base_dataset = base_dataset
        self.k = k
        self.base_len = len(base_dataset)

    def __len__(self):
        return self.base_len * (self.k + 1)

    def __getitem__(self, idx):
        # Map the expanded index back into the actual base subset index boundaries
        actual_idx = idx % self.base_len
        return self.base_dataset[actual_idx]

# dataset的建立用來訓練讀檔的

# 先建立 base_dataset，用來讀取所有圖片路徑與 label
# 這裡 transform=None，因為只是用來取得 label 和 class 資訊
base_dataset = datasets.ImageFolder(
    root=train_dir,
    transform=None
)

# 取得每張圖片的 label
targets = np.array(base_dataset.targets)

# 建立所有圖片的 index
indices = np.arange(len(base_dataset))

# 將 train 資料切成 train / validation
# stratify=targets 代表切分時盡量維持每個類別的比例
train_idx, val_idx = train_test_split(
    indices,
    test_size=0.2,
    random_state=42,
    stratify=targets
)

# train_full 使用 train_transform
# 這裡會套用資料增強和影像預處理
train_full = datasets.ImageFolder(
    root=train_dir,
    transform=train_transform
)

# val_full 使用 val_test_transform
# validation 不使用隨機資料增強，只做固定預處理
val_full = datasets.ImageFolder(
    root=train_dir,
    transform=val_test_transform
)

# Create the actual base train and validation subsets
base_train_dataset = Subset(train_full, train_idx)
val_dataset = Subset(val_full, val_idx)

def calculate_class_weights(subset, num_classes):
    # Extract the targets of the subset images
    indices = subset.indices
    targets = [subset.dataset.targets[i] for i in indices]
    
    # Count occurrences of each class index
    class_counts = Counter(targets)
    total_samples = len(targets)
    
    # Compute inverse frequency weights
    weights = []
    for i in range(num_classes):
        count = class_counts.get(i, 1) # avoid division by zero
        weight = total_samples / (num_classes * count)
        weights.append(weight)
        
    return torch.tensor(weights, dtype=torch.float)

# Compute the weights dynamically based on your actual 18,336 split images
num_classes = len(base_dataset.classes)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
class_weights = calculate_class_weights(base_train_dataset, num_classes).to(device)

print("Calculated Class Weights:")
for name, w in zip(base_dataset.classes, class_weights):
    print(f"  {name}: {w.item():.4f}")

# Expand the training dataset by k
K_EXPANSION = 0  # k=1 doubles dataset size, k=2 triples it, etc.
train_dataset = DatasetMultiplier(base_train_dataset, k=K_EXPANSION)

# Re-establish DataLoaders
train_loader = DataLoader(
    train_dataset,
    batch_size=32,
    shuffle=True,
    num_workers=0
)

val_loader = DataLoader(
    val_dataset,
    batch_size=32,
    shuffle=False,
    num_workers=0
)

print("========== Expanded Dataset Details ==========")
print("Original Split Train Images :", len(base_train_dataset))
print(f"Multiplier Factor (k)       : {K_EXPANSION}")
print("Final Augmented Train Images:", len(train_dataset)) # Will print 36672 if k=1
print("Validation Images           :", len(val_dataset))

Calculated Class Weights:
  Atelectasis: 0.4985
  Cardiomegaly: 1.1845
  Consolidation: 0.7958
  Edema: 1.3735
  Effusion: 0.7707
  Emphysema: 1.5129
  Fibrosis: 2.5467
  Hernia: 21.4456
  Infiltration: 0.5262
  Mass: 0.8477
  No Finding: 0.5095
  Nodule: 0.8436
  Pleural_Thickening: 2.1296
  Pneumonia: 7.5457
  Pneumothorax: 1.0667
========== Expanded Dataset Details ==========
Original Split Train Images : 18336
Multiplier Factor (k)       : 0
Final Augmented Train Images: 18336
Validation Images           : 4585


In [5]:
# 1. Device Configuration (Utilize GPU if available)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

# 2. Model Definition (ResNet50)
# Pretrained weights provide a robust feature extraction foundation for chest X-rays
model = models.resnet50(weights=models.ResNet50_Weights.DEFAULT)

# Modify the final fully connected layer to output the correct number of classes (15)
# Add a Dropout layer before the final fully connected layer to prevent co-adaptation of features
num_classes = len(base_dataset.classes) 
model.fc = nn.Sequential(
    nn.Dropout(p=0.5),
    nn.Linear(model.fc.in_features, num_classes)
)
model = model.to(device)

# Initially freeze ALL layers in the ResNet50 backbone (everything except the new model.fc head)
for name, param in model.named_parameters():
    if "fc" not in name:
        param.requires_grad = False

# 3. Loss Function & Optimization Configuration
criterion = nn.CrossEntropyLoss(weight=class_weights)
# Optimizer initially ONLY updates the unfrozen classification head (model.fc)
optimizer = optim.AdamW(model.fc.parameters(), lr=1e-3, weight_decay=1e-3)
# Automatically drops learning rate by half if validation loss does not improve for 2 epochs
scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='min', factor=0.5, patience=2)

print(f"Model successfully loaded. Class count: {num_classes}")

Using device: cuda
Model successfully loaded. Class count: 15


In [6]:
# 4. Training and Validation Process
num_epochs = 20
best_val_acc = 0.0
backbone_unfrozen = False

for epoch in range(num_epochs):
    
    # After 2 epochs of warming up the classification head, unfreeze the entire backbone
    if epoch == 2 and not backbone_unfrozen:
        print("\n>>> Warm-up completed! Unfreezing ResNet50 backbone layers for fine-tuning...")
        for param in model.parameters():
            param.requires_grad = True
            
        # Re-initialize optimizer with a safe, lower learning rate for full-network fine-tuning
        optimizer = optim.AdamW(model.parameters(), lr=2e-5, weight_decay=1e-3)
        # Re-bind the scheduler to the updated optimizer configuration
        scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='min', factor=0.5, patience=2)
        backbone_unfrozen = True

    # --- Training Phase ---
    model.train()
    running_loss = 0.0
    correct_train = 0
    total_train = 0
    
    train_bar = tqdm(train_loader, desc=f"Epoch [{epoch+1}/{num_epochs}] Training")
    for images, labels in train_bar:
        images, labels = images.to(device), labels.to(device)
        
        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        
        running_loss += loss.item() * images.size(0)
        _, predicted = torch.max(outputs, 1)
        total_train += labels.size(0)
        correct_train += (predicted == labels).sum().item()
        
        train_bar.set_postfix(loss=loss.item())
        
    epoch_train_loss = running_loss / len(train_loader.dataset)
    epoch_train_acc = correct_train / total_train
    
    # --- Validation Phase ---
    model.eval()
    val_loss = 0.0
    correct_val = 0
    total_val = 0
    
    with torch.no_grad():
        for images, labels in val_loader:
            images, labels = images.to(device), labels.to(device)
            outputs = model(images)
            loss = criterion(outputs, labels)
            
            val_loss += loss.item() * images.size(0)
            _, predicted = torch.max(outputs, 1)
            total_val += labels.size(0)
            correct_val += (predicted == labels).sum().item()
            
    epoch_val_loss = val_loss / len(val_loader.dataset)
    epoch_val_acc = correct_val / total_val
    
    # Step the learning rate scheduler based on current validation loss metrics
    scheduler.step(epoch_val_loss)
    
    print(f"Epoch [{epoch+1}/{num_epochs}] Results:")
    print(f"  Train Loss: {epoch_train_loss:.4f} | Train Acc: {epoch_train_acc*100:.2f}%")
    print(f"  Val Loss: {epoch_val_loss:.4f} | Val Acc: {epoch_val_acc*100:.2f}%")
    
    # Checkpoint evaluation & saving the best model weights
    if epoch_val_acc > best_val_acc:
        best_val_acc = epoch_val_acc
        torch.save(model.state_dict(), "best_resnet50.pth")
        print("  => New best validation accuracy! Saved model weights.")

print(f"\nTraining completed. Highest Validation Accuracy achieved: {best_val_acc*100:.2f}%")

Epoch [1/20] Training: 100%|██████████| 573/573 [04:22<00:00,  2.18it/s, loss=2.59]


Epoch [1/20] Results:
  Train Loss: 2.6445 | Train Acc: 13.41%
  Val Loss: 2.5502 | Val Acc: 16.90%
  => New best validation accuracy! Saved model weights.


Epoch [2/20] Training: 100%|██████████| 573/573 [04:06<00:00,  2.32it/s, loss=2.58]


Epoch [2/20] Results:
  Train Loss: 2.5863 | Train Acc: 15.92%
  Val Loss: 2.5206 | Val Acc: 18.69%
  => New best validation accuracy! Saved model weights.

>>> Warm-up completed! Unfreezing ResNet50 backbone layers for fine-tuning...


Epoch [3/20] Training: 100%|██████████| 573/573 [04:33<00:00,  2.10it/s, loss=2.48]


...

Epoch [18/20] Results:
  Train Loss: 1.8386 | Train Acc: 33.02%
  Val Loss: 2.1892 | Val Acc: 29.71%


Epoch [19/20] Training: 100%|██████████| 573/573 [04:37<00:00,  2.06it/s, loss=2.32] 


Epoch [19/20] Results:
  Train Loss: 1.7870 | Train Acc: 33.91%
  Val Loss: 2.1802 | Val Acc: 30.64%
  => New best validation accuracy! Saved model weights.


Epoch [20/20] Training: 100%|██████████| 573/573 [04:39<00:00,  2.05it/s, loss=1.46] 


Epoch [20/20] Results:
  Train Loss: 1.7525 | Train Acc: 34.42%
  Val Loss: 2.2050 | Val Acc: 30.58%

Training completed. Highest Validation Accuracy achieved: 30.64%


In [7]:
# 5. Custom Test Dataset Loader Implementation
class XRayTestDataset(Dataset):
    def __init__(self, test_dir, transform=None):
        self.test_dir = test_dir
        self.transform = transform
        files = os.listdir(test_dir)
        
        # Sort files numerically (0.jpg, 1.jpg...) to maintain sequential structural consistency
        try:
            self.filenames = sorted(files, key=lambda x: int(os.path.splitext(x)[0]))
        except ValueError:
            self.filenames = sorted(files)

    def __len__(self):
        return len(self.filenames)

    def __getitem__(self, idx):
        filename = self.filenames[idx]
        img_path = os.path.join(self.test_dir, filename)
        img = Image.open(img_path).convert("RGB")
        if self.transform:
            img = self.transform(img)
        return img, filename

# Instantiate Test DataLoader
test_dataset = XRayTestDataset(test_dir=test_dir, transform=val_test_transform)
test_loader = DataLoader(test_dataset, batch_size=32, shuffle=False, num_workers=0)

# Load the best-saved model checkpoint parameters
model.load_state_dict(torch.load("best_resnet50.pth"))
model.eval()

submission_data = []
class_names = base_dataset.classes # Pull string mapping directly from ImageFolder classes

# --- Inference / Prediction Generation Phase ---
print("Starting testing inference on test set files...")
with torch.no_grad():
    for images, filenames in tqdm(test_loader, desc="Testing Inference"):
        images = images.to(device)
        outputs = model(images)
        _, predicted = torch.max(outputs, 1)
        
        for i in range(len(filenames)):
            filename = filenames[i]
            pred_class_idx = predicted[i].item()
            pred_class_name = class_names[pred_class_idx]
            
            # Formulate tracking index match format
            row_id = len(submission_data)
            submission_data.append({
                "id": row_id,
                "filename": filename,
                "label": pred_class_name
            })

# 6. Structuring into DataFrame and Exporting Target File
submission_df = pd.DataFrame(submission_data)
submission_df.to_csv("submission.csv", index=False)
print("Submission file generation successfully concluded. Output saved as 'submission.csv'!")

Starting testing inference on test set files...


Testing Inference: 100%|██████████| 157/157 [01:03<00:00,  2.47it/s]

Submission file generation successfully concluded. Output saved as 'submission.csv'!
